In [24]:
import polars as pl

# batch 8

In [25]:
import extern
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt >/tmp/runs")

''

In [26]:
prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
prev.shape

(63406, 1)

In [27]:
total = pl.read_csv('runlists/acc_less_than_20k_mbytes.csv', has_header=False)
total.columns = ["acc"]
total.shape

(710928, 1)

In [28]:
batch8_possible = total.join(prev, on="acc", how="anti")
batch8_possible.shape

(647522, 1)

In [29]:
# batch8 = batch8_possible.sample(100000)
# print(batch8.shape, batch8[:5])
# batch8.write_csv('runlists/batch8_100000.txt', include_header=False)

In [30]:
# 
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt runlists/batch8_100000.txt >/tmp/runs")

prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
print(prev.shape)

batch9_possible = total.join(prev, on="acc", how="anti")
batch9_possible.shape

(163406, 1)


(547522, 1)

In [31]:
# batch9_possible.write_csv('runlists/batch9_da_rest.txt', include_header=False)

# batch10 - cleaning up and everythin 10Gbp or less

In [32]:
extern.run('cat  s3_us-east2.txt s3_woodcrob-sandpiper-us-east-1.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev1 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev1.columns = ["acc", "time", "size", "path"]
prev2 = prev1.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0)).select(["acc", "size"])
prev2.shape, prev2[:3]

((137852, 2),
 shape: (3, 2)
 ┌───────────┬───────┐
 │ acc       ┆ size  │
 │ ---       ┆ ---   │
 │ str       ┆ i64   │
 ╞═══════════╪═══════╡
 │ DRR001961 ┆ 26759 │
 │ DRR003630 ┆ 60975 │
 │ DRR003636 ┆ 45914 │
 └───────────┴───────┘)

In [33]:
prev2.filter(pl.col("size") == 0).shape

(4305, 2)

In [34]:
prev3 = prev2.filter(pl.col("size") > 0)
prev3.shape

(133547, 2)

In [35]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [36]:
batch10_possible = df.filter(pl.col('mbases') < 8000).join(prev, on="acc", how="anti")
batch10_possible.shape, batch10_possible.sample(3)

((471178, 4),
 shape: (3, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR12126885 ┆ 2020-08-01T00:00:00+00:00 ┆ 1719   ┆ 514    │
 │ SRR30564806 ┆ 2024-09-05T00:00:00+00:00 ┆ 2795   ┆ 809    │
 │ SRR1765647  ┆ 2015-01-17T00:00:00+00:00 ┆ 778    ┆ 465    │
 └─────────────┴───────────────────────────┴────────┴────────┘)

In [41]:
batch10_possible.select('acc').write_csv('runlists/batch10_10gbp_or_smaller.txt', include_header=False)

# batch11 small fast test of multiple

In [38]:
per_acc = pl.read_csv('~/m/msingle/mess/174_R220_renew/processing_20240531/per_acc_summary.csv')
per_acc.shape, per_acc[:3]

((245436, 11),
 shape: (3, 11)
 ┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
 │ sample     ┆ root_cove ┆ species_c ┆ bacterial ┆ … ┆ warning ┆ predictio ┆ organism  ┆ host_or_n │
 │ ---        ┆ rage      ┆ overage   ┆ _archaeal ┆   ┆ ---     ┆ n         ┆ ---       ┆ ot        │
 │ str        ┆ ---       ┆ ---       ┆ _bases    ┆   ┆ str     ┆ ---       ┆ str       ┆ ---       │
 │            ┆ f64       ┆ f64       ┆ ---       ┆   ┆         ┆ str       ┆           ┆ str       │
 │            ┆           ┆           ┆ i64       ┆   ┆         ┆           ┆           ┆           │
 ╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
 │ SRR8634435 ┆ 345.4     ┆ 0.933237  ┆ 117232064 ┆ … ┆ null    ┆ host      ┆ feces met ┆ host      │
 │            ┆           ┆           ┆ 2         ┆   ┆         ┆           ┆ agenome   ┆           │
 │ SRR8640623 ┆ 730.5     ┆ 0.127748  ┆ 139594647 ┆

In [39]:
batch11 = batch10_possible.join(per_acc.filter(pl.col('root_coverage')>0), left_on='acc', right_on='sample', how='inner').sort('mbases').head(4)

In [40]:
batch10.select('acc').write_csv('runlists/batch11_4.txt', include_header=False)